# LV4 - nadzirano učenje - linearni modeli i vrednovanje

- nadzirano učenje = određivanje ovisnosti između ulaznih veličina X i izlazne veličine y na temelju podatkovnih primjera
- nama će u ovoj vježbi y biti kontinuirana veličina (neki realni broj) -> regresija
- model = aproksimacija funkcionalne ovisnosti između X i y = funkcija s konačnim brojem parametara theta
- postupak učenja = određivanje parametara theta na temelju podatkovnih primjera
- za aproksimaciju te ovisnosti koristi se neka aproksimacijska funkcija. mi ćemo koristiti linearnu funkciju -> linearna regresija

In [1]:
from sklearn.linear_model import LinearRegression


- kako bismo mogli provesti postupak učenja, potrebni su nam X i y
- međutim, nakon učenja morat ćemo nekako provjeriti je li model kvalitetan, tj. je li dobro odredio ovisnost između X i y
- stoga ćemo model podijeliti na skup za učenje i skup za testiranje
- uobičajeno je uzeti 20% podataka za testiranje

In [3]:
from sklearn.model_selection import train_test_split

- kako bismo dobili bolje rezultate, bilo bi dobro prije učenja skalirati podatke - svesti ih na isti interval
- time model neće favorizirati značajke s većim vrijednostima naspram manjih (npr. godina rođenja i broj djece), a i model će biti brži
- možemo koristiti min-max ili standardizaciju
- min-max skaliranje je jednostavno, skalira podatke najčešće na interval od 0 do 1. međutim, ako u datasetu imamo velike outliere, tada će nam se "normalni" podaci zbiti u sredinu što nije idealno
- standardizacija za skaliranje koristi standardnu devijaciju i dat će nam bolji raspon vrijednosti

In [4]:
from sklearn.preprocessing import MinMaxScaler, StandardScaler

- osim skaliranja numeričkih značajki, moramo i kodirati kategoričke (nominalne i ordinalne) značajke
- modelu ništa ne znače vrijednosti Audi, Ford, Nissan... i on ne može s njima dobro raditi (a neki modeli ne mogu uopće)
- ali zato možemo od jedne značajke npr. Make napraviti onoliko značajki koliko ima marki automobila: Make_Audi, Make_Ford... i dodijeliti im true/false vrijednosti
- tada model može jako lako raditi s ovim vrijednostima

In [7]:
from sklearn.preprocessing import OneHotEncoder
# ili
import pandas as pd # funkcija get_dummies

sve zajedno:

In [11]:
data = pd.read_csv('data_C02_emission.csv')

# one-hot-encoding
data = pd.get_dummies(data, columns=['Make', 'Model', 'Vehicle Class', 'Transmission', 'Fuel Type'])

# određivanje ulaznih i izlazne varijable
X = data.drop(columns='CO2 Emissions (g/km)')
y = data['CO2 Emissions (g/km)']

# podjela na train i test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

# skaliranje numeričkih značajki
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# stvaranje i učenje modela
model = LinearRegression().fit(X_train, y_train)

# predviđanje vrijednosti na skupu za treniranje
y_test_predicted = model.predict(X_test)

### Vrednovanje modela

- sad kad smo napravili model, moramo nekako odrediti je li on dobar, tj. predviđa li dobro izlaznu varijablu na temelju dosad neviđenih ulaznih podataka
- za to imamo različite metrike:

1. MSE - srednja kvadratna pogreška
- srednja vrijednost kvadriranih razlika stvarne i predviđene vrijednosti y
- kvadriramo razliku kako bismo izbjegli negativne vrijednosti, ali i kako bismo dodatno kaznili velika odstupanja od stvarnih vrijednosti
- ova metrika je korisna, ali nije nam osobito čitljiva - ako predviđamo cijenu kuće, što nam znači eur na kvadrat?

In [12]:
from sklearn.metrics import mean_squared_error
mean_squared_error(y_test, y_test_predicted)

20.029399834964615

2. RMSE - korijen srednje kvadratne pogreške
- ime kaže što radi
- korisnija metrika od MSE, jer ju možemo lakše interpretirati - vidimo iz nje koliko je u prosjeku model odstupao od stvarne vrijednosti

In [13]:
from sklearn.metrics import root_mean_squared_error
root_mean_squared_error(y_test, y_test_predicted)

4.475421749395761

3. MAE - srednja apsolutna pogreška
- slična MSE, ali umjesto kvadriranja, uzima apsolutnu vrijednost razlike stvarne i predviđene vrijednosti
- time ne kažnjava outliere, pa usporedbom MAE i RMSE možemo otprilike vidjeti je li model imao puno velikih pogreški

In [14]:
from sklearn.metrics import mean_absolute_error
mean_absolute_error(y_test, y_test_predicted)

2.5556850704375447

4. MAPE - srednja apsolutna postotna pogreška
- mjeri prosječnu relativnu pogrešku modela u postocima:  
- MAPE od npr. 0.1 znači da model u prosjeku griješi 10% u odnosu na stvarne vrijednosti
- međutim, MAPE nije dobar kada su vrijednosti izlazne varijable male, a pogotovo kasnije za klasifikaciju kad imamo npr. klase 0 i 1 zbog dijeljenja s nulom
- tada može dati jako veliku grešku iako to ustvari možda nije istina

In [15]:
from sklearn.metrics import mean_absolute_percentage_error
mean_absolute_percentage_error(y_test, y_test_predicted)

0.01069583986817783

5. R2 - koeficijent determinacije
- pokazuje koliko je varijacija u podacima naš model obuhvatio - koliko promjene u izlaznoj varijabli naši ulazni podaci mogu objasniti
- savršen model ima R2 = 1 (100%), a loši modeli mogu imati i negativan R2

In [16]:
from sklearn.metrics import r2_score
r2_score(y_test, y_test_predicted)

0.9948435895958436